# 🏦 Bank X — Real-time Fraud Monitoring Dashboard

Reads Spark Streaming outputs from `/workspace/data/tp5/output`.

- **Last 20 users** seen in recent transactions
- **Last 10 seconds** of activity
- Windowed metrics (3h / 7d / 3w / 3mo) + lifetime aggregates
- Auto-refresh every 5 seconds

In [ ]:
import sys
import importlib
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display
import ipywidgets as widgets

warnings.filterwarnings("ignore")

sys.path.insert(0, "/home/jovyan/work/tp5")
import dashboard_io as dio
importlib.reload(dio)
from dashboard_io import read_latest_parquet, flatten_window_columns

# #region agent log
dio._dbg("A", "notebook_cell1_import", {"flatten_module": dio.flatten_window_columns.__module__})
# #endregion

OUTPUT_BASE = Path("/workspace/data/tp5/output")
RECENT_PATH = OUTPUT_BASE / "recent_transactions"
LIFETIME_PATH = OUTPUT_BASE / "lifetime"
WINDOW_PATHS = {
    "3_hours": OUTPUT_BASE / "windowed" / "3_hours",
    "7_days": OUTPUT_BASE / "windowed" / "7_days",
    "3_weeks": OUTPUT_BASE / "windowed" / "3_weeks",
    "3_months": OUTPUT_BASE / "windowed" / "3_months",
}
REFRESH_SECONDS = 5

In [ ]:
from IPython.display import HTML

DASHBOARD_CSS = """
<style>
:root {
  --bx-bg: #0f172a;
  --bx-surface: #ffffff;
  --bx-muted: #f1f5f9;
  --bx-border: #e2e8f0;
  --bx-text: #0f172a;
  --bx-subtle: #64748b;
  --bx-primary: #2563eb;
  --bx-accent: #6366f1;
  --bx-success: #10b981;
  --bx-warn: #f59e0b;
  --bx-danger: #ef4444;
  --bx-shadow: 0 6px 20px rgba(15, 23, 42, 0.08);
}

.bx-wrap {
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Inter, sans-serif;
  color: var(--bx-text);
  padding: 4px 2px 16px;
}

.bx-header {
  background: linear-gradient(135deg, #1e3a8a 0%, #2563eb 50%, #6366f1 100%);
  color: #fff;
  border-radius: 14px;
  padding: 18px 22px;
  box-shadow: var(--bx-shadow);
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 18px;
}
.bx-header h1 {
  font-size: 20px;
  margin: 0 0 4px;
  font-weight: 700;
  letter-spacing: 0.2px;
}
.bx-header .bx-sub {
  font-size: 13px;
  opacity: 0.9;
}
.bx-pulse {
  display: inline-block;
  width: 10px; height: 10px;
  border-radius: 50%;
  background: #34d399;
  box-shadow: 0 0 0 0 rgba(52, 211, 153, 0.7);
  animation: bxpulse 1.6s infinite;
  margin-right: 8px;
}
@keyframes bxpulse {
  0%   { box-shadow: 0 0 0 0 rgba(52, 211, 153, 0.7); }
  70%  { box-shadow: 0 0 0 12px rgba(52, 211, 153, 0); }
  100% { box-shadow: 0 0 0 0 rgba(52, 211, 153, 0); }
}

.bx-kpis {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 14px;
  margin-bottom: 18px;
}
.bx-kpi {
  background: var(--bx-surface);
  border: 1px solid var(--bx-border);
  border-radius: 12px;
  padding: 14px 16px;
  box-shadow: var(--bx-shadow);
  position: relative;
  overflow: hidden;
}
.bx-kpi::before {
  content: "";
  position: absolute;
  left: 0; top: 0; bottom: 0;
  width: 4px;
  background: var(--bx-primary);
}
.bx-kpi.success::before { background: var(--bx-success); }
.bx-kpi.warn::before    { background: var(--bx-warn); }
.bx-kpi.danger::before  { background: var(--bx-danger); }
.bx-kpi.accent::before  { background: var(--bx-accent); }
.bx-kpi .bx-label {
  font-size: 11px;
  text-transform: uppercase;
  letter-spacing: 0.8px;
  color: var(--bx-subtle);
  font-weight: 600;
}
.bx-kpi .bx-value {
  font-size: 22px;
  font-weight: 700;
  margin-top: 4px;
  color: var(--bx-text);
}

.bx-card {
  background: var(--bx-surface);
  border: 1px solid var(--bx-border);
  border-radius: 12px;
  padding: 14px 16px 6px;
  box-shadow: var(--bx-shadow);
  margin-bottom: 16px;
}
.bx-card h3 {
  font-size: 14px;
  margin: 0 0 10px;
  color: var(--bx-text);
  text-transform: uppercase;
  letter-spacing: 0.6px;
  font-weight: 700;
  border-bottom: 2px solid var(--bx-muted);
  padding-bottom: 8px;
  display: flex;
  align-items: center;
  gap: 8px;
}
.bx-card h3 .bx-dot {
  width: 8px; height: 8px; border-radius: 50%;
  background: var(--bx-primary);
}

.bx-card table {
  border-collapse: collapse;
  width: 100%;
  font-size: 12.5px;
}
.bx-card table thead th {
  background: #f8fafc;
  color: var(--bx-subtle);
  text-transform: uppercase;
  font-size: 10.5px;
  letter-spacing: 0.6px;
  text-align: left;
  padding: 8px 10px;
  border-bottom: 1px solid var(--bx-border);
  font-weight: 700;
}
.bx-card table tbody td {
  padding: 7px 10px;
  border-bottom: 1px solid #f1f5f9;
}
.bx-card table tbody tr:hover td {
  background: #f8fafc;
}
.bx-card table tbody tr:last-child td { border-bottom: none; }

.bx-user-grid {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(140px, 1fr));
  gap: 6px;
}
.bx-chip {
  background: var(--bx-muted);
  color: var(--bx-text);
  padding: 6px 10px;
  border-radius: 999px;
  font-size: 12px;
  text-align: center;
  border: 1px solid var(--bx-border);
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
}

.bx-waiting {
  background: #fef3c7;
  border: 1px solid #fcd34d;
  color: #92400e;
  padding: 14px 16px;
  border-radius: 10px;
  font-weight: 600;
}
</style>
"""

display(HTML(DASHBOARD_CSS))

In [ ]:
# read_latest_parquet + flatten_window_columns imported from tp5/dashboard_io.py (cell 1)


def anomaly_score(row) -> float:
    """Simple heuristic: high amount vs rolling average."""
    avg = row.get("lifetime_avg_amount", row.get("avg_amount", 0)) or 1
    amt = row.get("amount", row.get("lifetime_avg_amount", 0))
    return float(amt) / float(avg) if avg else 0.0


_TABLE_STYLES = [
    {"selector": "table", "props": "border-collapse: collapse; width: 100%; font-size: 12.5px;"},
    {"selector": "thead th", "props": (
        "background: #f8fafc; color: #64748b; text-transform: uppercase;"
        "font-size: 10.5px; letter-spacing: 0.6px; text-align: left;"
        "padding: 8px 10px; border-bottom: 1px solid #e2e8f0; font-weight: 700;"
    )},
    {"selector": "tbody td", "props": "padding: 7px 10px; border-bottom: 1px solid #f1f5f9;"},
    {"selector": "tbody tr:hover td", "props": "background: #f8fafc;"},
]


def style_anomalies(df: pd.DataFrame, amount_col: str = "amount") -> object:
    if df.empty:
        return df
    styled = df.style.set_table_styles(_TABLE_STYLES).hide(axis="index")
    if amount_col in df.columns and "lifetime_avg_amount" in df.columns:
        def highlight(row):
            score = anomaly_score(row)
            if score >= 5:
                return ["background-color: #fee2e2; color: #991b1b; font-weight: 600;"] * len(row)
            if score >= 2:
                return ["background-color: #fef3c7; color: #92400e;"] * len(row)
            return [""] * len(row)
        styled = styled.apply(highlight, axis=1)
    return styled


def style_table(df: pd.DataFrame) -> object:
    if df.empty:
        return df
    return df.style.set_table_styles(_TABLE_STYLES).hide(axis="index")

In [ ]:
import numpy as np
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch

# ---------------------------------------------------------------------------
# Visual theme for matplotlib charts
# ---------------------------------------------------------------------------
CHART_PALETTE = {
    "primary":   "#2563eb",
    "accent":    "#6366f1",
    "success":   "#10b981",
    "warn":      "#f59e0b",
    "danger":    "#ef4444",
    "text":      "#0f172a",
    "subtle":    "#64748b",
    "grid":      "#e2e8f0",
    "bg":        "#ffffff",
    "soft":      "#f8fafc",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelcolor": CHART_PALETTE["subtle"],
    "axes.edgecolor": CHART_PALETTE["grid"],
    "axes.labelweight": "600",
    "xtick.color": CHART_PALETTE["subtle"],
    "ytick.color": CHART_PALETTE["subtle"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": CHART_PALETTE["bg"],
    "axes.facecolor": CHART_PALETTE["bg"],
})


def _style_ax(ax, title=None):
    if title:
        ax.set_title(title, color=CHART_PALETTE["text"], pad=12, loc="left")
    ax.grid(axis="y", linestyle="--", alpha=0.45, color=CHART_PALETTE["grid"])
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color(CHART_PALETTE["grid"])


def _annotate_bars(ax, fmt="{:,.0f}", offset=3, color=None):
    color = color or CHART_PALETTE["text"]
    for patch in ax.patches:
        h = patch.get_height()
        if h == 0 or np.isnan(h):
            continue
        ax.annotate(
            fmt.format(h),
            xy=(patch.get_x() + patch.get_width() / 2, h),
            xytext=(0, offset), textcoords="offset points",
            ha="center", va="bottom",
            fontsize=9, fontweight="700", color=color,
        )


def _annotate_hbars(ax, fmt="{:,.0f}"):
    for patch in ax.patches:
        w = patch.get_width()
        if w == 0 or np.isnan(w):
            continue
        ax.annotate(
            fmt.format(w),
            xy=(w, patch.get_y() + patch.get_height() / 2),
            xytext=(4, 0), textcoords="offset points",
            ha="left", va="center",
            fontsize=9, fontweight="700", color=CHART_PALETTE["text"],
        )


def render_charts(last_10s: pd.DataFrame, time_col: str):
    """Draw a 2x2 panel of professional charts from the last-10s activity."""
    if last_10s.empty or "amount" not in last_10s.columns:
        return

    fig = plt.figure(figsize=(13, 7.5))
    gs = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.28)
    ax_tl  = fig.add_subplot(gs[0, :])   # timeline full width
    ax_dir = fig.add_subplot(gs[1, 0])
    ax_top = fig.add_subplot(gs[1, 1])

    # --- 1) Timeline: tx count + volume per second ----------------------
    df = last_10s.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.set_index(time_col).sort_index()
    per_sec = df.resample("1s").agg(
        tx=("amount", "count"),
        volume=("amount", "sum"),
    ).fillna(0)

    ax_tl.fill_between(
        per_sec.index, per_sec["volume"],
        color=CHART_PALETTE["primary"], alpha=0.15, zorder=1,
    )
    ax_tl.plot(
        per_sec.index, per_sec["volume"],
        color=CHART_PALETTE["primary"], linewidth=2.2,
        marker="o", markersize=5, markerfacecolor="white",
        markeredgewidth=2, markeredgecolor=CHART_PALETTE["primary"],
        label="Volume (MRU)", zorder=3,
    )
    ax_tl2 = ax_tl.twinx()
    ax_tl2.bar(
        per_sec.index, per_sec["tx"],
        width=0.0006, color=CHART_PALETTE["accent"], alpha=0.35,
        label="Tx count", zorder=2,
    )
    ax_tl2.set_ylabel("Tx count", color=CHART_PALETTE["accent"])
    ax_tl2.tick_params(axis="y", colors=CHART_PALETTE["accent"])
    ax_tl2.spines["top"].set_visible(False)
    ax_tl2.spines["right"].set_color(CHART_PALETTE["grid"])
    ax_tl2.grid(False)

    _style_ax(ax_tl, "Activity timeline — volume & transactions / second")
    ax_tl.set_ylabel("Volume (MRU)", color=CHART_PALETTE["primary"])
    ax_tl.tick_params(axis="y", colors=CHART_PALETTE["primary"])
    ax_tl.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    fig.autofmt_xdate(rotation=0, ha="center")

    lines1, labels1 = ax_tl.get_legend_handles_labels()
    lines2, labels2 = ax_tl2.get_legend_handles_labels()
    ax_tl.legend(
        lines1 + lines2, labels1 + labels2,
        loc="upper left", frameon=False, fontsize=9, ncol=2,
    )

    # --- 2) Volume by direction (bar) -----------------------------------
    if "direction" in df.columns:
        by_dir = df.groupby("direction")["amount"].sum().sort_values(ascending=False)
        colors_dir = [
            CHART_PALETTE["success"] if d == "received" else CHART_PALETTE["primary"]
            for d in by_dir.index
        ]
        bars = ax_dir.bar(
            by_dir.index, by_dir.values,
            color=colors_dir, edgecolor="white", linewidth=2, width=0.55,
        )
        _style_ax(ax_dir, "Volume by direction")
        ax_dir.set_ylabel("MRU")
        ax_dir.set_xlabel("")
        _annotate_bars(ax_dir, fmt="{:,.2f}")
        ax_dir.margins(y=0.18)
    else:
        ax_dir.axis("off")

    # --- 3) Top 10 users by volume (horizontal) -------------------------
    if "user_id" in df.columns:
        top_users = (
            df.groupby("user_id")["amount"].sum()
            .sort_values(ascending=True).tail(10)
        )
        cmap = plt.cm.get_cmap("Blues")
        colors_u = [cmap(0.35 + 0.55 * i / max(len(top_users) - 1, 1)) for i in range(len(top_users))]
        ax_top.barh(
            top_users.index, top_users.values,
            color=colors_u, edgecolor="white", linewidth=1.2,
        )
        _style_ax(ax_top, "Top 10 users by volume")
        ax_top.set_xlabel("MRU")
        ax_top.grid(axis="x", linestyle="--", alpha=0.45, color=CHART_PALETTE["grid"])
        ax_top.grid(axis="y", visible=False)
        ax_top.tick_params(axis="y", labelsize=9)
        _annotate_hbars(ax_top, fmt="{:,.2f}")
        ax_top.margins(x=0.18)
    else:
        ax_top.axis("off")

    fig.suptitle("")  # keep clean; cards already provide titles
    plt.tight_layout()
    plt.show()


def _kpi(label: str, value, kind: str = "") -> str:
    return (
        f'<div class="bx-kpi {kind}">'
        f'<div class="bx-label">{label}</div>'
        f'<div class="bx-value">{value}</div>'
        f'</div>'
    )


def _card_open(title: str) -> str:
    return f'<div class="bx-card"><h3><span class="bx-dot"></span>{title}</h3>'


def _card_close() -> str:
    return "</div>"


def render_dashboard():
    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(seconds=10)

    recent = read_latest_parquet(RECENT_PATH, limit_files=50)
    lifetime = read_latest_parquet(LIFETIME_PATH, limit_files=10)

    display(HTML(DASHBOARD_CSS))
    display(HTML('<div class="bx-wrap">'))

    header_html = (
        '<div class="bx-header">'
        '<div>'
        '<h1>🏦 Bank X — Real-time Fraud Monitoring</h1>'
        f'<div class="bx-sub"><span class="bx-pulse"></span>Live · {now.strftime("%Y-%m-%d %H:%M:%S UTC")}</div>'
        '</div>'
        '<div style="text-align:right; font-size:12px; opacity:0.9;">'
        f'Auto-refresh every {REFRESH_SECONDS}s'
        '</div>'
        '</div>'
    )
    display(HTML(header_html))

    if recent.empty:
        display(HTML(
            '<div class="bx-waiting">⏳ Waiting for data… '
            'Start the stack with <code>docker compose up -d</code></div></div>'
        ))
        return

    recent["event_time"] = pd.to_datetime(recent["event_time"], utc=True)
    if "ingested_at" in recent.columns:
        recent["ingested_at"] = pd.to_datetime(recent["ingested_at"], utc=True)
        time_col = "ingested_at"
    else:
        time_col = "event_time"

    last_10s = recent[recent[time_col] >= cutoff].copy()
    if last_10s.empty:
        last_10s = recent.nlargest(30, time_col).copy()

    last_users = (
        recent.sort_values(time_col, ascending=False)["user_id"]
        .drop_duplicates()
        .head(20)
        .tolist()
    )

    total_amount = float(last_10s["amount"].sum()) if "amount" in last_10s.columns else 0.0
    tx_count = len(last_10s)
    unique_users = last_10s["user_id"].nunique() if "user_id" in last_10s.columns else 0
    max_amount = float(last_10s["amount"].max()) if "amount" in last_10s.columns and not last_10s.empty else 0.0

    kpis_html = (
        '<div class="bx-kpis">'
        + _kpi("Transactions (10s)", f"{tx_count:,}", "")
        + _kpi("Active users (10s)", f"{unique_users:,}", "accent")
        + _kpi("Total volume (MRU)", f"{total_amount:,.2f}", "success")
        + _kpi("Max single tx", f"{max_amount:,.2f}", "warn")
        + '</div>'
    )
    display(HTML(kpis_html))

    # --- Analytics panel (charts) ------------------------------------------
    display(HTML(_card_open("Analytics · last 10 seconds")))
    render_charts(last_10s, time_col)
    display(HTML(_card_close()))

    display(HTML(_card_open(f"Last 10 seconds activity (by {time_col})")))
    cols = ["event_time", "user_id", "direction", "counterparty", "amount", "tx_id"]
    if "ingested_at" in last_10s.columns:
        cols = ["ingested_at"] + cols
    cols = [c for c in cols if c in last_10s.columns]
    show_10s = last_10s[cols].sort_values(time_col, ascending=False) if not last_10s.empty else pd.DataFrame(columns=cols)
    display(style_anomalies(show_10s.head(50)))
    display(HTML(_card_close()))

    display(HTML(_card_open("Last 20 active users")))
    chips = "".join(f'<div class="bx-chip">{u}</div>' for u in last_users)
    display(HTML(f'<div class="bx-user-grid">{chips}</div>'))
    display(HTML(_card_close()))

    user_recent = recent[recent["user_id"].isin(last_users)].copy()
    if not lifetime.empty and not user_recent.empty:
        if "processed_at" in lifetime.columns:
            lifetime = lifetime.sort_values("processed_at", ascending=False).drop_duplicates(
                subset=["user_id", "direction"], keep="first"
            )
        merged = user_recent.merge(lifetime, on=["user_id", "direction"], how="left")
        display(HTML(_card_open("Recent transactions + lifetime metrics (anomaly highlighting)")))
        display(style_anomalies(merged.sort_values(time_col, ascending=False).head(40)))
        display(HTML(_card_close()))

    for win_name, path in WINDOW_PATHS.items():
        wdf = flatten_window_columns(read_latest_parquet(path, limit_files=15))
        if wdf.empty:
            continue
        wdf = wdf[wdf["user_id"].isin(last_users)] if "user_id" in wdf.columns else wdf
        display(HTML(_card_open(f"Window · {win_name.replace('_', ' ')}")))
        sort_col = "processed_at" if "processed_at" in wdf.columns else wdf.columns[0]
        display(style_table(wdf.sort_values(sort_col, ascending=False).head(30)))
        display(HTML(_card_close()))

    display(HTML('</div>'))

In [ ]:
import time

auto = widgets.Checkbox(value=True, description="Auto-refresh (5s)")
display(auto)

try:
    while auto.value:
        clear_output(wait=True)
        display(auto)
        render_dashboard()
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Stopped auto-refresh.")

Checkbox(value=True, description='Auto-refresh (5s)')

HTML(value='<h2>Dashboard @ 2026-05-21T14:09:36.062413+00:00</h2>')

HTML(value='<h3>Last 10 seconds activity (by ingested_at)</h3>')

,ingested_at,event_time,user_id,direction,counterparty,amount,tx_id
0,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:09+00:00,user_b_3924,received,client_3490,1.160000,61a9c748-8a6a-4692-b2d4-59b0bf71e3e1
1,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:12+00:00,user_b_2126,received,user_a_388,12.340000,64b4e273-686f-4266-80fa-6fcf52f9969e
28,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:25+00:00,client_4448,sent,client_2944,2.180000,97b59a74-e596-4972-bc84-a8f92557fac6
27,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:22+00:00,user_b_3826,sent,user_a_122,198.710000,6a7d7510-ab59-4f4a-be05-33633fe6dd6b
26,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:16+00:00,client_3292,sent,client_2670,18.690000,a298c972-2af5-441e-98bf-c8622ec8a678
25,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:15+00:00,user_a_2148,sent,user_a_1225,167.790000,451c73f7-822c-4370-9999-f59c4717d76b
24,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:15+00:00,user_a_2148,sent,client_2287,23.630000,bb7854ff-6371-489a-bc1d-151fa385f4e4
23,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:14+00:00,user_b_1865,sent,user_b_1409,3.190000,9130a5b6-c5e6-44e7-a3db-2fbfc20cc008
22,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:12+00:00,user_a_388,sent,user_b_2126,12.340000,64b4e273-686f-4266-80fa-6fcf52f9969e
21,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:09+00:00,client_3490,sent,user_b_3924,1.160000,61a9c748-8a6a-4692-b2d4-59b0bf71e3e1


HTML(value='<h3>Last 20 active users</h3>')

,user_id
0,user_b_3924
1,user_a_197
2,user_a_4944
3,client_4938
4,user_b_2422
5,user_a_1916
6,user_a_2897
7,user_b_3501
8,client_2058
9,client_4461


HTML(value='<h3>Window: 3_hours</h3>')

,user_id,direction,avg_amount,tx_count,total_amount,distinct_counterparties,window_name,processed_at,window_start,window_end
2148,user_b_2422,received,5.86,1,5.86,1,3_hours,2026-05-21 13:35:53.188,1779359100000000000,1779369900000000000
2373,user_b_3501,received,3.18,1,3.18,1,3_hours,2026-05-21 13:35:53.188,1779359100000000000,1779369900000000000
2461,user_b_3501,received,3.18,1,3.18,1,3_hours,2026-05-21 13:35:53.188,1779359040000000000,1779369840000000000
3669,user_b_2422,received,5.86,1,5.86,1,3_hours,2026-05-21 13:35:53.188,1779359040000000000,1779369840000000000
